In [ ]:
from google.colab import drive

drive.mount('/content/gdrive')

In [ ]:
from pathlib import Path

ruta = Path("/content/gdrive/MyDrive/2026-0")

print("Existe:", ruta.exists())

for x in ruta.iterdir():
    print(x.name)

In [ ]:
import os
from pathlib import Path
from PIL import Image
from torchvision.transforms import functional as F
from torchvision.transforms import InterpolationMode

# ==========================================================
# RUTAS
# ==========================================================

INPUT_ROOT = Path("/content/gdrive/MyDrive/2026-0/DatasetOriginal")
OUTPUT_ROOT = Path("/content/gdrive/MyDrive/2026-0/TrashNet_transformaciones")

CLASSES = [
    "cardboard",
    "glass",
    "metal",
    "paper",
    "plastic",
    "trash"
]

# ==========================================================
# NIVELES EXPERIMENTALES
# ==========================================================

BRIGHTNESS = {
    "low": 1.2,
    "medium": 1.4,
    "high": 1.6
}

CONTRAST = {
    "low": 1.2,
    "medium": 1.4,
    "high": 1.6
}

ROTATION = {
    "low": 10,
    "medium": 30,
    "high": 60
}

ZOOM = {
    "low": 1.10,
    "medium": 1.25,
    "high": 1.50
}

# ==========================================================
# FUNCIÓN DE ZOOM DETERMINISTA
# ==========================================================

def zoom_image(img, zoom_factor):
    """
    Aplica zoom-in manteniendo el tamaño final original.

    Ejemplo:
    zoom_factor = 1.25 -> ampliación de 25 %
    """

    width, height = img.size

    # Tamaño de la región que se conservará antes de reescalar
    crop_width = int(width / zoom_factor)
    crop_height = int(height / zoom_factor)

    # Coordenadas para crop central
    left = (width - crop_width) // 2
    top = (height - crop_height) // 2
    right = left + crop_width
    bottom = top + crop_height

    # Recortar la región central
    img_crop = img.crop((left, top, right, bottom))

    # Regresar al tamaño original
    img_zoom = img_crop.resize(
        (width, height),
        resample=Image.Resampling.BILINEAR
    )

    return img_zoom


# ==========================================================
# CREAR DIRECTORIOS
# ==========================================================

transformations = {
    "brightness": BRIGHTNESS,
    "contrast": CONTRAST,
    "rotation": ROTATION,
    "zoom": ZOOM
}

for transformation, levels in transformations.items():
    for level in levels.keys():
        for class_name in CLASSES:
            output_dir = (
                OUTPUT_ROOT /
                f"{transformation}_{level}" /
                class_name
            )
            output_dir.mkdir(
                parents=True,
                exist_ok=True
            )


# ==========================================================
# APLICAR TRANSFORMACIONES
# ==========================================================

valid_extensions = [".jpg", ".jpeg", ".png", ".bmp"]

for class_name in CLASSES:

    class_dir = INPUT_ROOT / class_name

    image_files = [
        p for p in class_dir.iterdir()
        if p.suffix.lower() in valid_extensions
    ]

    print(f"Procesando {class_name}: {len(image_files)} imágenes")

    for image_path in image_files:

        img = Image.open(image_path).convert("RGB")

        # --------------------------------------------------
        # BRILLO
        # --------------------------------------------------

        for level, factor in BRIGHTNESS.items():

            transformed = F.adjust_brightness(
                img,
                brightness_factor=factor
            )

            output_path = (
                OUTPUT_ROOT /
                f"brightness_{level}" /
                class_name /
                image_path.name
            )

            transformed.save(output_path)

        # --------------------------------------------------
        # CONTRASTE
        # --------------------------------------------------

        for level, factor in CONTRAST.items():

            transformed = F.adjust_contrast(
                img,
                contrast_factor=factor
            )

            output_path = (
                OUTPUT_ROOT /
                f"contrast_{level}" /
                class_name /
                image_path.name
            )

            transformed.save(output_path)

        # --------------------------------------------------
        # ROTACIÓN
        # --------------------------------------------------

        for level, angle in ROTATION.items():

            transformed = F.rotate(
                img,
                angle=angle,
                interpolation=InterpolationMode.BILINEAR,
                expand=False,
                fill=0
            )

            output_path = (
                OUTPUT_ROOT /
                f"rotation_{level}" /
                class_name /
                image_path.name
            )

            transformed.save(output_path)

        # --------------------------------------------------
        # ZOOM
        # --------------------------------------------------

        for level, zoom_factor in ZOOM.items():

            transformed = zoom_image(
                img,
                zoom_factor
            )

            output_path = (
                OUTPUT_ROOT /
                f"zoom_{level}" /
                class_name /
                image_path.name
            )

            transformed.save(output_path)

print("\nTransformaciones completadas.")